# 1120 56일차

## 1. BatchNormalization

층을 지난 값을 배치 단위로 평균 0, 분산 1로 맞추는 층 (`keras81_BatchNormalization1`)

입력을 스케일링해서 넣어도 층을 지나는 동안 값의 분포가 다시 흩어짐. 뒤쪽 층은 매번 달라지는 분포에 맞춰 다시 배워야 해서 학습이 느려짐

1. `from tensorflow.keras.layers import BatchNormalization` : 다른 층과 같은 자리에서 불러옴
2. `model.add(BatchNormalization())` : 층과 층 사이에 넣음. 인자 없이 씀
3. 배치 안의 값으로 평균과 분산을 구해 표준화함

- **하는 일은 Standardization. 데이터가 아니라 층을 지난 값에 적용함**
- `StandardScaler`는 훈련 전에 데이터 전체로 한 번, BN은 훈련 중에 층마다 배치마다 다시 맞춤
- 배치 안의 값으로 통계를 내므로 `batch_size`가 작으면 평균과 분산이 흔들림
- Conv 층 뒤와 Dense 층 뒤 어디에든 넣을 수 있음

## 2. BN과 활성화 함수 순서

BN을 활성화 함수 앞에 둘지 뒤에 둘지 (`keras81_BatchNormalization1`, `keras81_BatchNormalization2`)

두 층의 순서만 바뀌는 것이지만, ReLU가 음수를 0으로 자르기 때문에 표준화에 들어가는 값이 달라짐

1. BN → ReLU : 표준화한 값을 ReLU에 넣음. 평균이 0이라 절반쯤이 0으로 잘림
2. ReLU → BN : 0으로 잘린 값을 표준화함. 0이 몰려 있는 분포를 평균 0으로 옮김

![BN과 ReLU 순서에 따른 값 분포](assets/bn-relu-order.svg)

| 순서 | 다음 층이 받는 값 |
|---|---|
| BN → ReLU | 평균 0으로 맞춘 뒤 자르므로 절반쯤이 정확히 0 |
| ReLU → BN | 0에 쌓인 값까지 같이 옮겨서, 0이던 값이 음수 자리로 감 |

- **어느 쪽이 낫다고 정해져 있지 않고, 데이터와 모델마다 다름**
- CNN 계열에서는 대체로 BN → ReLU가 나았음
- `Conv2D(..., activation='relu')`로 적으면 활성화가 층 안에서 끝나므로, 뒤에 `BatchNormalization()`을 두면 ReLU → BN이 됨
- BN을 먼저 두려면 활성화를 `activation=None`으로 빼고 `Activation('relu')`를 따로 쌓음
- 순서를 비교할 때는 층 구성 말고 나머지 조건을 똑같이 둠

## 3. XOR 문제

직선 하나로는 0과 1을 가를 수 없는 문제 (`m11_1_and` ~ `m11_4_xor_unsolved`)

퍼셉트론은 입력에 가중치를 곱해 더한 값을 기준과 비교해 0과 1로 가름. 기준이 직선 하나라서, 직선으로 갈라지지 않는 문제는 가중치를 어떻게 잡아도 못 맞춤

| 입력 | AND | OR | XOR |
|---|---|---|---|
| 0, 0 | 0 | 0 | 0 |
| 0, 1 | 0 | 1 | 1 |
| 1, 0 | 0 | 1 | 1 |
| 1, 1 | 1 | 1 | 0 |

1. AND : 둘 다 1일 때만 1. 1인 점이 한 귀퉁이에만 있어 직선 하나로 갈림
2. OR : 하나라도 1이면 1. 0인 점이 한 귀퉁이에만 있어 직선 하나로 갈림
3. XOR : 두 입력이 다를 때만 1. 1인 점 둘이 서로 마주 보는 귀퉁이에 놓여 직선으로 갈리지 않음

![AND·OR·XOR의 네 점과 경계선](assets/xor-linear-separability.svg)

- **XOR은 직선을 어디에 그어도 네 점 중 셋까지만 맞아서 정답률이 75%를 넘지 못함**
- 입력이 두 개라 네 가지 경우가 사각형의 네 귀퉁이에 놓임
- 같은 대각선에 놓인 두 점이 서로 다른 정답이면 직선 하나로는 나눌 수 없음
- `Perceptron`과 `LinearSVC`는 둘 다 직선 하나로 가르는 모델이라 AND·OR만 맞힘
- `Dense` 한 층으로 바꿔도 같음. 경계가 직선 하나라서 epochs를 올려도 달라지지 않음
- 이 한계가 밝혀지면서 AI 1차 겨울이 옴

#### cf) 75%와 실제 결과

- 75%는 선을 가장 잘 그었을 때의 상한이고, 학습이 그 자리로 간다는 뜻은 아님
- XOR을 `Perceptron`·`LinearSVC`로 돌리면 네 점을 모두 0으로 찍어 정답률이 0.5로 나옴
- 상한은 그릴 수 있는 선 중 가장 좋은 것, 결과는 손실을 줄이다 멈춘 자리라 둘이 다름